# Abundance Of The Selected Cluster Representatives

These are new mappings to `all_assembled_top_contigs_tot.fasta`, using the same up-to-2-million cleaned read pairs as the full catalogue (all cleaned paired and orphan reads in isolate mode). Each row represents one selected 95/85 ANI cluster's representative. No additional top-100 cap or viral-evidence filter is applied.

The heatmap uses `log10(1 + raw RPKM)` with average-linkage hierarchical clustering on Euclidean distances, on both axes. It is not row-scaled: the colours retain abundance differences. The negative control, when present, remains visible. Only representatives with non-zero remapped RPKM are plotted; all representatives remain in the exported matrices.

Remapping can assign reads to these representatives that previously mapped to other members. These measurements are not summed member abundances and must not be confused with the original full-catalogue RPKM retained in the selection/annotation metadata. Reference-set-specific CoverM RPKM values can change when the mapping reference pool changes.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from IPython.display import display

SAMPLES=list(snakemake.params.samples)
sns.set_style("ticks", {'axes.grid': True})
sns.set_palette("colorblind")
plt.rcParams['svg.fonttype']='none'
plt.rcParams['font.family']='sans-serif'
plt.rcParams['font.sans-serif']=['Liberation Sans']
plt.rcParams['axes.linewidth']=1.5
plt.rcParams['savefig.dpi']=300


## Raw RPKM, Counts, Breadth And Mean Depth

Read the same eight CoverM fields as MOSAIC's existing normalisation notebook. Mean depth comes directly from CoverM, not read count divided by contig length. Export raw measurements without changing the general Python abundance-normalisation step.


In [ ]:
coverm_tables={}
for sample, path in zip(SAMPLES, snakemake.input.covstats):
    frame=pd.read_csv(path, sep="\t")
    frame.columns=["contig_id", "mean_depth", "length_bp", "covered_bases",
                   "mapped_reads", "variance", "trimmed_mean", "RPKM_raw"]
    frame=frame.set_index("contig_id")
    frame["breadth_percent"]=(100 * frame["covered_bases"] / frame["length_bp"].replace(0, np.nan)).fillna(0)
    coverm_tables[sample]=frame

matrices={}
for field, output in [("RPKM_raw", snakemake.output.rpkm),
                      ("mapped_reads", snakemake.output.counts),
                      ("breadth_percent", snakemake.output.breadth),
                      ("covered_bases", snakemake.output.covered_bases),
                      ("mean_depth", snakemake.output.depth)]:
    matrix=pd.DataFrame({sample: coverm_tables[sample][field] for sample in SAMPLES}).fillna(0)
    matrix.index.name="assembled_contig"
    matrix.to_csv(output)
    matrices[field]=matrix
rpkm=matrices["RPKM_raw"]
display(rpkm)


## Cluster Representative Raw-RPKM Clustermap

Representatives on the y axis; samples on the x axis. Dendrograms group similar remapped abundance profiles, not nucleotide sequence similarity.


In [ ]:
plot_rpkm=rpkm.loc[rpkm.gt(0).any(axis=1)].copy()
if not plot_rpkm.empty:
    grid=sns.clustermap(np.log10(1 + plot_rpkm), method="average", metric="euclidean",
        row_cluster=len(plot_rpkm) > 1, col_cluster=len(SAMPLES) > 1,
        cmap="mako", vmin=0, xticklabels=True, yticklabels=True,
        figsize=(min(30, max(12, 7 + 0.7 * len(SAMPLES))), min(80, max(8, 3 + 0.22 * len(plot_rpkm)))),
        cbar_kws={"label": "log10(1 + raw RPKM)"})
    grid.ax_heatmap.set_xlabel("Sample", fontsize=16)
    grid.ax_heatmap.set_ylabel("Selected cluster representative", fontsize=16)
    grid.ax_heatmap.tick_params(axis="x", labelrotation=90, labelsize=10)
    grid.ax_heatmap.tick_params(axis="y", labelsize=8)
    grid.fig.suptitle("Selected cluster representatives: remapped raw RPKM", y=1.01, fontsize=18)
    grid.savefig(snakemake.output.png, dpi=300, bbox_inches="tight")
    grid.savefig(snakemake.output.svg, bbox_inches="tight")
    plt.show()
    plt.close(grid.fig)
else:
    fig, ax=plt.subplots(figsize=(12, 6))
    ax.text(0.5, 0.5, "No non-zero abundance in the remapped selected set", ha="center", va="center")
    ax.set_axis_off()
    fig.savefig(snakemake.output.png, dpi=300, bbox_inches="tight")
    fig.savefig(snakemake.output.svg, bbox_inches="tight")
    plt.show()
    plt.close(fig)
